# 🧹 Atomic Step 2: Dynamic Noise Suppression

This notebook loads the standardized 16kHz mono WAV audio, applies non-stationary noise reduction using the `noisereduce` library to strip away background hum, chatter, and hiss, and exports the clean WAV file.

In [ ]:
# Install dependencies
!pip install -q noisereduce librosa soundfile
print("[SUCCESS] Dependencies installed!")

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
try:
    drive.mount('/content/drive')
    print("[SUCCESS] Google Drive mounted.")
except Exception:
    print("[INFO] Already mounted or skipped.")

## ⚙️ Parameters

In [ ]:
# @markdown ### 📁 Audio Paths
standardized_audio_path = "/content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Atomic Notebooks/Standardized_Audio/MarauliKhurad1_standardized.wav" # @param {type:"string"}
cleaned_audio_folder = "/content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Atomic Notebooks/Cleaned_Audio/" # @param {type:"string"}

if not os.path.exists(standardized_audio_path):
    print(f"[ERROR] Standardized audio not found at: '{standardized_audio_path}'")
else:
    audio_filename = os.path.basename(standardized_audio_path)
    # Remove '_standardized' suffix from target name if present
    audio_name_only = audio_filename.replace("_standardized.wav", "").replace(".wav", "")
    os.makedirs(cleaned_audio_folder, exist_ok=True)
    print(f"[SUCCESS] Validated path. Cleaned audio will be saved in: {cleaned_audio_folder}")

## 🧹 Process Denoising

In [ ]:
import numpy as np
# Runtime compatibility patch for older SciPy versions loaded with NumPy 2.x
if not hasattr(np._core._multiarray_umath, "_blas_supports_fpe"):
    setattr(np._core._multiarray_umath, "_blas_supports_fpe", lambda x: False)

import librosa
import soundfile as sf
import noisereduce as nr

if os.path.exists(standardized_audio_path):
    print(f"Loading standardized audio: '{standardized_audio_path}'")
    y, sr = librosa.load(standardized_audio_path, sr=16000, mono=True)
    
    print("Applying Non-Stationary Noise Reduction...")
    # Apply noise reduction (stationary=False adapts to changing background noise)
    y_denoised = nr.reduce_noise(y=y, sr=sr, stationary=False, prop_decrease=0.85)
    
    # Save clean WAV file
    cleaned_audio_path = os.path.join(cleaned_audio_folder, f"{audio_name_only}_cleaned.wav")
    sf.write(cleaned_audio_path, y_denoised, sr)
    
    print(f"\n[SUCCESS] Noise suppression complete!")
    print(f"Cleaned audio successfully stored at: '{cleaned_audio_path}'")